# HW3: NER с трансформерными моделями

**Задача:** Дообучение трансформерной модели для NER-задачи на русском языке

**Датасет:** factRuEval-2016 (gusevski/factrueval2016)

**Базовая модель:** cointegrated/rubert-tiny2

---

## План работы

1. **Базовое обучение NER** — загрузка данных, препроцессинг, baseline
2. **MLM предобучение** — адаптация модели к домену через Masked Language Modeling
3. **Улучшенное маскирование** — WholeWordMask и Entity masking
4. **Синтетическая разметка** — генерация pseudo-labels через DeepPavlov NER
5. **Сравнение подходов** — финальные выводы

---

## 0. Настройка окружения и импорты

In [1]:
# Установка зависимостей (для Colab/Kaggle)
# !pip install transformers datasets seqeval accelerate deeppavlov -q

In [1]:
# Фиксируем random_state для воспроизводимости
RANDOM_STATE = 42

import warnings
warnings.filterwarnings('ignore')

import os
import numpy as np
import pandas as pd
from tqdm.auto import tqdm
from collections import Counter

# Фиксируем seeds
np.random.seed(RANDOM_STATE)

import torch
torch.manual_seed(RANDOM_STATE)

from torch.utils.data import DataLoader

def get_device():
    """
    Автоматическое определение устройства.
    Приоритет: CUDA -> MPS (Apple Silicon) -> CPU
    """
    if torch.cuda.is_available():
        device = torch.device('cuda')
        torch.cuda.manual_seed_all(RANDOM_STATE)
        print(f'GPU: {torch.cuda.get_device_name(0)}')
    elif hasattr(torch.backends, 'mps') and torch.backends.mps.is_available():
        device = torch.device('mps')
        print('Using Apple Silicon MPS')
    else:
        device = torch.device('cpu')
        print('Using CPU')
    return device

DEVICE = get_device()

# Параметры обучения, адаптированные под устройство
if DEVICE.type == 'cuda':
    BATCH_SIZE = 16
    EVAL_BATCH_SIZE = 32
    FP16 = True  # mixed precision для GPU
elif DEVICE.type == 'mps':
    BATCH_SIZE = 8
    EVAL_BATCH_SIZE = 16
    FP16 = False  # MPS не поддерживает fp16 полноценно
else:  # CPU
    BATCH_SIZE = 8
    EVAL_BATCH_SIZE = 8
    FP16 = False

print(f'Device: {DEVICE}')
print(f'Batch size: {BATCH_SIZE}, Eval batch size: {EVAL_BATCH_SIZE}, FP16: {FP16}')

Using Apple Silicon MPS
Device: mps
Batch size: 8, Eval batch size: 16, FP16: False


In [2]:
from datasets import load_dataset, DatasetDict, Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForTokenClassification,
    AutoModelForMaskedLM,
    DataCollatorForTokenClassification,
    DataCollatorForLanguageModeling,
    DataCollatorForWholeWordMask,
    TrainingArguments,
    Trainer,
    set_seed
)
from seqeval.metrics import (
    classification_report as seqeval_report,
    f1_score as seqeval_f1,
    precision_score as seqeval_precision,
    recall_score as seqeval_recall
)

set_seed(RANDOM_STATE)

## 1. Загрузка и анализ датасета

**factRuEval-2016** — эталонный корпус для оценки NER на русском языке.

Сущности:
- **PER** (Person) — имена людей
- **ORG** (Organization) — названия организаций  
- **LOC** (Location) — географические объекты

In [3]:
# Загрузка датасета
# Датасет имеет нестандартную структуру: данные упакованы в поле 'data' как список словарей
raw_dataset = load_dataset('gusevski/factrueval2016')
print('Исходная структура:')
print(raw_dataset)

# Распаковываем данные из поля 'data' в нормальный формат
def unpack_dataset(raw_split):
    """Распаковка данных из поля 'data' в отдельные примеры."""
    # data[0] содержит список всех примеров
    examples = raw_split['data'][0]
    return Dataset.from_list(examples)

dataset = DatasetDict({
    'train': unpack_dataset(raw_dataset['train']),
    'validation': unpack_dataset(raw_dataset['validation']),
    'test': unpack_dataset(raw_dataset['test'])
})

print('\nРаспакованная структура:')
print(dataset)

Repo card metadata block was not found. Setting CardData to empty.


Исходная структура:
DatasetDict({
    train: Dataset({
        features: ['data'],
        num_rows: 1
    })
    validation: Dataset({
        features: ['data'],
        num_rows: 1
    })
    test: Dataset({
        features: ['data'],
        num_rows: 1
    })
})

Распакованная структура:
DatasetDict({
    train: Dataset({
        features: ['id', 'tokens', 'length', 'ner_tags_str', 'ner_tags'],
        num_rows: 7746
    })
    validation: Dataset({
        features: ['id', 'tokens', 'length', 'ner_tags_str', 'ner_tags'],
        num_rows: 2582
    })
    test: Dataset({
        features: ['id', 'tokens', 'length', 'ner_tags_str', 'ner_tags'],
        num_rows: 2582
    })
})


In [4]:
# Анализ структуры данных
print('Пример из train:')
example = dataset['train'][0]
print(f"ID: {example['id']}")
print(f"Tokens ({len(example['tokens'])}): {example['tokens'][:10]}...")
print(f"NER tags (str): {example['ner_tags_str'][:10]}...")
print(f"NER tags (int): {example['ner_tags'][:10]}...")

Пример из train:
ID: 0
Tokens (47): ['"', 'Если', 'Миронов', 'занял', 'столь', 'оппозиционную', 'позицию', ',', 'то', 'мне']...
NER tags (str): ['O', 'O', 'B-PER', 'O', 'O', 'O', 'O', 'O', 'O', 'O']...
NER tags (int): [0, 0, 1, 0, 0, 0, 0, 0, 0, 0]...


In [5]:
# Получаем список меток из данных
# Собираем уникальные метки из ner_tags_str
all_tags = set()
for example in dataset['train']:
    all_tags.update(example['ner_tags_str'])

# Сортируем метки: O первым, затем B-*, I-* в алфавитном порядке
label_list = sorted(all_tags, key=lambda x: (x != 'O', x))
print(f'Метки NER ({len(label_list)}): {label_list}')

# Создаём маппинги
label2id = {label: i for i, label in enumerate(label_list)}
id2label = {i: label for i, label in enumerate(label_list)}

print(f'\nlabel2id: {label2id}')

Метки NER (7): ['O', 'B-LOC', 'B-ORG', 'B-PER', 'I-LOC', 'I-ORG', 'I-PER']

label2id: {'O': 0, 'B-LOC': 1, 'B-ORG': 2, 'B-PER': 3, 'I-LOC': 4, 'I-ORG': 5, 'I-PER': 6}


### BIO-схема разметки

| Метка | Значение | Описание |
|-------|----------|----------|
| **B-PER** | **B**eginning of Person | Начало сущности "персона" |
| **I-PER** | **I**nside Person | Продолжение сущности "персона" |
| **B-ORG** | **B**eginning of Organization | Начало сущности "организация" |
| **I-ORG** | **I**nside Organization | Продолжение сущности "организация" |
| **B-LOC** | **B**eginning of Location | Начало сущности "локация" |
| **I-LOC** | **I**nside Location | Продолжение сущности "локация" |
| **O** | Outside | Не является сущностью |

In [6]:
# Примеры BIO-разметки из датасета
print('ПРИМЕРЫ BIO-РАЗМЕТКИ')
print('='*70)

examples_shown = 0
for example in dataset['train']:
    tokens = example['tokens']
    tags = example['ner_tags_str']
    
    # Ищем многословные сущности (B-* + I-*)
    i = 0
    while i < len(tags):
        if tags[i].startswith('B-'):
            ent_type = tags[i][2:]  # PER, ORG, LOC
            start_i = i
            i += 1
            # Собираем все I-* токены
            while i < len(tags) and tags[i] == f'I-{ent_type}':
                i += 1
            end_i = i
            
            # Показываем только многословные сущности (минимум 2 токена)
            if end_i - start_i >= 2 and examples_shown < 8:
                # Контекст: 2 слова до и после
                ctx_start = max(0, start_i - 2)
                ctx_end = min(len(tokens), end_i + 2)
                
                print(f'\nСущность: "{" ".join(tokens[start_i:end_i])}" ({ent_type})')
                print(f'Токены:   {tokens[start_i:end_i]}')
                print(f'Метки:    {tags[start_i:end_i]}')
                print(f'Контекст: ...{" ".join(tokens[ctx_start:ctx_end])}...')
                examples_shown += 1
        else:
            i += 1
    
    if examples_shown >= 8:
        break

print('\n' + '='*70)
print('Каждая сущность начинается с B-*, продолжается I-* того же типа')

ПРИМЕРЫ BIO-РАЗМЕТКИ

Сущность: "Единой Россией" (ORG)
Токены:   ['Единой', 'Россией']
Метки:    ['B-ORG', 'I-ORG']
Контекст: ...сегодня " Единой Россией '' и...

Сущность: "Ханты-Мансийском автономном округе" (LOC)
Токены:   ['Ханты-Мансийском', 'автономном', 'округе']
Метки:    ['B-LOC', 'I-LOC', 'I-LOC']
Контекст: ...В Ханты-Мансийском автономном округе с должности...

Сущность: "Николай Гудожников" (PER)
Токены:   ['Николай', 'Гудожников']
Метки:    ['B-PER', 'I-PER']
Контекст: ...начальник УВД Николай Гудожников ....

Сущность: "Российского научного центра" (ORG)
Токены:   ['Российского', 'научного', 'центра']
Метки:    ['B-ORG', 'I-ORG', 'I-ORG']
Контекст: ...является президентом Российского научного центра " Курчатовский...

Сущность: "Красненькая речка" (LOC)
Токены:   ['Красненькая', 'речка']
Метки:    ['B-LOC', 'I-LOC']
Контекст: ...округе " Красненькая речка '' ....

Сущность: "Карл Айкенберри" (PER)
Токены:   ['Карл', 'Айкенберри']
Метки:    ['B-PER', 'I-PER']
Контекст: ...

In [7]:
# Статистика по сущностям: считаем и токены, и сущности

def count_statistics(dataset_split):
    """
    Подсчёт статистики по сущностям в датасете.
    
    Возвращает:
    - entity_counts: количество сущностей (считаем только B-* теги)
    - token_counts: количество токенов с разметкой (B-* и I-*)
    """
    entity_counts = Counter()  # количество сущностей
    token_counts = Counter()   # количество токенов
    
    for example in dataset_split:
        for tag in example['ner_tags_str']:
            if tag.startswith('B-'):
                # Начало новой сущности
                entity_type = tag[2:]  # B-PER -> PER
                entity_counts[entity_type] += 1
                token_counts[entity_type] += 1
            elif tag.startswith('I-'):
                # Продолжение сущности (только токен, не новая сущность)
                entity_type = tag[2:]  # I-PER -> PER
                token_counts[entity_type] += 1
    
    return entity_counts, token_counts

print('СТАТИСТИКА ПО ДАТАСЕТУ')
print('='*70)

for split in dataset:
    entity_counts, token_counts = count_statistics(dataset[split])
    total_entities = sum(entity_counts.values())
    total_tokens = sum(token_counts.values())
    
    print(f'\n{split.upper()} ({len(dataset[split])} примеров)')
    print('-'*50)
    print(f'{"Тип":<6} {"Сущностей":>12} {"Токенов":>12} {"Ср. длина":>12}')
    print('-'*50)
    
    for ent_type in ['PER', 'ORG', 'LOC']:
        ent_count = entity_counts[ent_type]
        tok_count = token_counts[ent_type]
        avg_len = tok_count / ent_count if ent_count > 0 else 0
        print(f'{ent_type:<6} {ent_count:>12} {tok_count:>12} {avg_len:>12.2f}')
    
    print('-'*50)
    total_avg = total_tokens / total_entities if total_entities > 0 else 0
    print(f'{"ВСЕГО":<6} {total_entities:>12} {total_tokens:>12} {total_avg:>12.2f}')

СТАТИСТИКА ПО ДАТАСЕТУ

TRAIN (7746 примеров)
--------------------------------------------------
Тип       Сущностей      Токенов    Ср. длина
--------------------------------------------------
PER            6359        10284         1.62
ORG            5084         8311         1.63
LOC            4335         5307         1.22
--------------------------------------------------
ВСЕГО         15778        23902         1.51

VALIDATION (2582 примеров)
--------------------------------------------------
Тип       Сущностей      Токенов    Ср. длина
--------------------------------------------------
PER            2110         3431         1.63
ORG            1629         2815         1.73
LOC            1394         1701         1.22
--------------------------------------------------
ВСЕГО          5133         7947         1.55

TEST (2582 примеров)
--------------------------------------------------
Тип       Сущностей      Токенов    Ср. длина
-----------------------------------------

---

## 2. Подготовка данных для обучения

### Выбор базовой модели

**cointegrated/rubert-tiny2** — компактный BERT для русского языка:
- 12 слоёв, 312 hidden size
- ~29M параметров (в 6 раз меньше BERT-base)
- Быстрое обучение и инференс
- Хорошее качество для своего размера

In [8]:
# Используем компактную модель для быстрого обучения
MODEL_NAME = 'cointegrated/rubert-tiny2'

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
print(f'Tokenizer: {tokenizer.__class__.__name__}')
print(f'Vocab size: {tokenizer.vocab_size}')

Tokenizer: BertTokenizerFast
Vocab size: 83828


### Токенизация с выравниванием меток

BERT использует subword токенизацию (WordPiece). Одно слово может разбиться на несколько токенов:

```
"Путин" -> ["Пу", "##тин"]
```

**Стратегия выравнивания:**
- Первый субтокен слова получает метку слова (B-PER, I-ORG и т.д.)
- Остальные субтокены получают `-100` (игнорируются при расчёте loss)
- Специальные токены [CLS], [SEP] также получают `-100`

In [9]:
def tokenize_and_align_labels(examples):
    """
    Токенизация с выравниванием NER-меток.
    
    Стратегия:
    - Первый субтокен слова получает метку слова
    - Остальные субтокены получают -100 (игнорируются в loss)
    - Специальные токены [CLS], [SEP] получают -100
    """
    tokenized = tokenizer(
        examples['tokens'],
        truncation=True,
        is_split_into_words=True,  # входные данные уже токенизированы по словам
        max_length=512
    )
    
    labels = []
    for i, ner_tags_str in enumerate(examples['ner_tags_str']):
        word_ids = tokenized.word_ids(batch_index=i)
        previous_word_idx = None
        label_ids_aligned = []
        
        for word_idx in word_ids:
            if word_idx is None:
                # Специальные токены [CLS], [SEP], [PAD]
                label_ids_aligned.append(-100)
            elif word_idx != previous_word_idx:
                # Первый субтокен слова — конвертируем строковую метку в ID
                tag_str = ner_tags_str[word_idx]
                label_ids_aligned.append(label2id[tag_str])
            else:
                # Продолжение слова (##token) — игнорируем
                label_ids_aligned.append(-100)
            
            previous_word_idx = word_idx
        
        labels.append(label_ids_aligned)
    
    tokenized['labels'] = labels
    return tokenized

In [10]:
# Применяем токенизацию к датасету
tokenized_dataset = dataset.map(
    tokenize_and_align_labels,
    batched=True,
    remove_columns=dataset['train'].column_names
)

print(tokenized_dataset)

Map: 100%|██████████| 2582/2582 [00:00<00:00, 17780.41 examples/s]

DatasetDict({
    train: Dataset({
        features: ['input_ids', 'token_type_ids', 'attention_mask', 'labels'],
        num_rows: 7746
    })
    validation: Dataset({
        features: ['input_ids', 'token_type_ids', 'attention_mask', 'labels'],
        num_rows: 2582
    })
    test: Dataset({
        features: ['input_ids', 'token_type_ids', 'attention_mask', 'labels'],
        num_rows: 2582
    })
})


In [11]:
# Проверяем выравнивание на примере
example_idx = 0
original = dataset['train'][example_idx]
tokenized = tokenized_dataset['train'][example_idx]

print('Исходные токены и метки:')
for token, tag in zip(original['tokens'], original['ner_tags_str']):
    print(f'  {token:20} -> {tag}')

print('\nПосле токенизации (субтокены):')
tokens = tokenizer.convert_ids_to_tokens(tokenized['input_ids'])
for token, label in zip(tokens[:30], tokenized['labels'][:30]):
    label_str = id2label[label] if label != -100 else '[IGNORE]'
    print(f'  {token:20} -> {label_str}')

Исходные токены и метки:
  "                    -> O
  Если                 -> O
  Миронов              -> B-PER
  занял                -> O
  столь                -> O
  оппозиционную        -> O
  позицию              -> O
  ,                    -> O
  то                   -> O
  мне                  -> O
  представляется       -> O
  ,                    -> O
  что                  -> O
  для                  -> O
  него                 -> O
  было                 -> O
  бы                   -> O
  порядочным           -> O
  и                    -> O
  правильным           -> O
  уйти                 -> O
  в                    -> O
  отставку             -> O
  с                    -> O
  занимаемого          -> O
  им                   -> O
  поста                -> O
  ,                    -> O
  поста                -> O
  ,                    -> O
  который              -> O
  предоставлен         -> O
  ему                  -> O
  сегодня              -> O
  "                

---

## 3. Baseline: NER без дообучения и с дообучением

### 3.1 Метрики оценки

Для NER используем **seqeval** — библиотеку для оценки последовательностей:
- **Precision** — доля правильно предсказанных сущностей среди всех предсказанных
- **Recall** — доля правильно найденных сущностей среди всех истинных
- **F1** — гармоническое среднее precision и recall

In [12]:
def compute_metrics(eval_pred):
    """
    Вычисление NER-метрик с использованием seqeval.
    """
    predictions, labels = eval_pred
    predictions = np.argmax(predictions, axis=2)
    
    # Убираем игнорируемые токены (-100) и конвертируем в строки
    true_labels = []
    true_predictions = []
    
    for pred_seq, label_seq in zip(predictions, labels):
        true_label_seq = []
        true_pred_seq = []
        
        for pred, label in zip(pred_seq, label_seq):
            if label != -100:
                true_label_seq.append(id2label[label])
                true_pred_seq.append(id2label[pred])
        
        true_labels.append(true_label_seq)
        true_predictions.append(true_pred_seq)
    
    return {
        'precision': seqeval_precision(true_labels, true_predictions),
        'recall': seqeval_recall(true_labels, true_predictions),
        'f1': seqeval_f1(true_labels, true_predictions)
    }

In [ ]:
def evaluate_model(model, dataset_split, description='Evaluating'):                                                                                  
      model.eval()                                                                                                                                     
      all_predictions = []                                                                                                                             
      all_labels = []                                                                                                                                  
                                                                                                                                                       
      # Определяем устройство модели                                                                                                                   
      model_device = next(model.parameters()).device                                                                                                   
                                                                                                                                                       
      data_collator = DataCollatorForTokenClassification(tokenizer)                                                                                    
      dataloader = DataLoader(                                                                                                                         
          dataset_split,                                                                                                                               
          batch_size=EVAL_BATCH_SIZE,                                                                                                                  
          collate_fn=data_collator                                                                                                                     
      )                                                                                                                                                
                                                                                                                                                       
      with torch.no_grad():                                                                                                                            
          for batch in tqdm(dataloader, desc=description):                                                                                             
              # Переносим на устройство МОДЕЛИ                                                                                                         
              batch = {k: v.to(model_device) for k, v in batch.items()}                                                                                
              outputs = model(**batch)                                                                                                                 
                                                                                                                                                       
              predictions = outputs.logits.argmax(dim=-1).cpu().numpy()                                                                                
              labels = batch['labels'].cpu().numpy()                                                                                                   
                                                                                                                                                       
              for pred, label in zip(predictions, labels):                                                                                             
                  valid_indices = label != -100                                                                                                        
                  all_predictions.append([id2label[p] for p in pred[valid_indices]])                                                                   
                  all_labels.append([id2label[l] for l in label[valid_indices]])                                                                       
                                                                                                                                                       
      results = {                                                                                                                                      
          'precision': seqeval_precision(all_labels, all_predictions),                                                                                 
          'recall': seqeval_recall(all_labels, all_predictions),                                                                                       
          'f1': seqeval_f1(all_labels, all_predictions)                                                                                                
      }                                                                                                                                                
                                                                                                                                                       
      print(f"\n{description}")                                                                                                                        
      print(f"Precision: {results['precision']:.4f}")                                                                                                  
      print(f"Recall:    {results['recall']:.4f}")                                                                                                     
      print(f"F1:        {results['f1']:.4f}")                                                                                                         
                                                                                                                                                       
      return results


### 3.2 Оценка ДО дообучения (zero-shot)

Загружаем предобученную модель со случайно инициализированной головой классификации и оцениваем качество "из коробки".

In [14]:
# Загружаем модель для NER (со случайной головой классификации)
model_baseline = AutoModelForTokenClassification.from_pretrained(
    MODEL_NAME,
    num_labels=len(label_list),
    id2label=id2label,
    label2id=label2id
).to(DEVICE)

print(f'Модель загружена: {MODEL_NAME}')
print(f'Количество параметров: {sum(p.numel() for p in model_baseline.parameters()):,}')

Some weights of BertForTokenClassification were not initialized from the model checkpoint at cointegrated/rubert-tiny2 and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Модель загружена: cointegrated/rubert-tiny2
Количество параметров: 29,098,303


In [15]:
# Оценка ДО дообучения
results_before = evaluate_model(
    model_baseline, 
    tokenized_dataset['test'],
    'Оценка ДО дообучения (zero-shot)'
)

Оценка ДО дообучения (zero-shot): 100%|██████████| 162/162 [00:04<00:00, 34.05it/s]



Оценка ДО дообучения (zero-shot)
Precision: 0.0156
Recall:    0.0795
F1:        0.0261

Детальный отчёт:
              precision    recall  f1-score   support

         LOC       0.02      0.07      0.03      1509
         ORG       0.01      0.04      0.01      1790
         PER       0.03      0.12      0.04      2187

   micro avg       0.02      0.08      0.03      5486
   macro avg       0.02      0.08      0.03      5486
weighted avg       0.02      0.08      0.03      5486



### 3.3 Дообучение baseline модели

**Гиперпараметры:**

| Параметр | Значение | Обоснование |
|----------|----------|-------------|
| learning_rate | 2e-5 | Стандарт для fine-tuning BERT |
| batch_size | 8-16 (авто) | Адаптируется под устройство (GPU: 16, CPU/MPS: 8) |
| epochs | 3 | Достаточно для NER на небольшом датасете |
| weight_decay | 0.01 | Регуляризация для предотвращения переобучения |
| warmup_ratio | 0.1 | Плавный старт обучения |
| fp16 | auto | Mixed precision на GPU для ускорения |

In [16]:
# Создаём свежую модель для дообучения
model_finetuned = AutoModelForTokenClassification.from_pretrained(
    MODEL_NAME,
    num_labels=len(label_list),
    id2label=id2label,
    label2id=label2id
).to(DEVICE)

Some weights of BertForTokenClassification were not initialized from the model checkpoint at cointegrated/rubert-tiny2 and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [17]:
# Настройка обучения
training_args = TrainingArguments(
    output_dir='./results/baseline',
    evaluation_strategy='epoch',
    save_strategy='epoch',
    learning_rate=2e-5,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=EVAL_BATCH_SIZE,
    num_train_epochs=3,
    weight_decay=0.01,
    warmup_ratio=0.1,
    logging_steps=50,
    load_best_model_at_end=True,
    metric_for_best_model='f1',
    fp16=FP16,
    seed=RANDOM_STATE,
    report_to='none',
    use_cpu=(DEVICE.type == 'cpu'),
)

data_collator = DataCollatorForTokenClassification(tokenizer)

In [18]:
trainer = Trainer(
    model=model_finetuned,
    args=training_args,
    train_dataset=tokenized_dataset['train'],
    eval_dataset=tokenized_dataset['test'],
    tokenizer=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics
)

# Обучение
trainer.train()

  2%|▏         | 51/2907 [00:05<03:22, 14.07it/s]

{'loss': 1.8165, 'grad_norm': 9.409748077392578, 'learning_rate': 3.436426116838488e-06, 'epoch': 0.05}


  4%|▎         | 102/2907 [00:09<03:48, 12.27it/s]

{'loss': 1.4607, 'grad_norm': 7.809354782104492, 'learning_rate': 6.872852233676976e-06, 'epoch': 0.1}


  5%|▌         | 153/2907 [00:13<02:45, 16.68it/s]

...

                                                   
100%|██████████| 2907/2907 [02:49<00:00, 19.61it/s]

{'eval_loss': 0.060797929763793945, 'eval_precision': 0.8640457222029788, 'eval_recall': 0.9094057601166606, 'eval_f1': 0.886145648312611, 'eval_runtime': 3.4774, 'eval_samples_per_second': 742.517, 'eval_steps_per_second': 46.587, 'epoch': 3.0}


100%|██████████| 2907/2907 [02:49<00:00, 17.13it/s]

{'train_runtime': 169.6866, 'train_samples_per_second': 136.947, 'train_steps_per_second': 17.132, 'train_loss': 0.1777650744676344, 'epoch': 3.0}


TrainOutput(global_step=2907, training_loss=0.1777650744676344, metrics={'train_runtime': 169.6866, 'train_samples_per_second': 136.947, 'train_steps_per_second': 17.132, 'total_flos': 16158516362976.0, 'train_loss': 0.1777650744676344, 'epoch': 3.0})

In [19]:
# Оценка ПОСЛЕ дообучения
results_after = evaluate_model(
    model_finetuned, 
    tokenized_dataset['test'],
    'Оценка ПОСЛЕ дообучения (baseline)'
)

Оценка ПОСЛЕ дообучения (baseline): 100%|██████████| 162/162 [00:02<00:00, 56.28it/s]



Оценка ПОСЛЕ дообучения (baseline)
Precision: 0.8640
Recall:    0.9094
F1:        0.8861

Детальный отчёт:
              precision    recall  f1-score   support

         LOC       0.91      0.92      0.92      1509
         ORG       0.77      0.83      0.80      1790
         PER       0.91      0.96      0.94      2187

   micro avg       0.86      0.91      0.89      5486
   macro avg       0.86      0.91      0.88      5486
weighted avg       0.87      0.91      0.89      5486



In [20]:
# Сравнение результатов
print('\n' + '='*50)
print('СРАВНЕНИЕ: ДО vs ПОСЛЕ дообучения')
print('='*50)
print(f"{'Метрика':<12} {'До':>10} {'После':>10} {'Δ':>10}")
print('-'*50)
for metric in ['precision', 'recall', 'f1']:
    before = results_before[metric]
    after = results_after[metric]
    delta = after - before
    print(f"{metric:<12} {before:>10.4f} {after:>10.4f} {delta:>+10.4f}")


СРАВНЕНИЕ: ДО vs ПОСЛЕ дообучения
Метрика              До      После          Δ
--------------------------------------------------
precision        0.0156     0.8640    +0.8485
recall           0.0795     0.9094    +0.8299
f1               0.0261     0.8861    +0.8601


---

## 4. MLM предобучение + NER

**Идея:** Перед NER-дообучением адаптируем модель к домену через Masked Language Modeling на train-части корпуса.

**Почему это работает:**
- Модель "привыкает" к лексике и стилю целевого корпуса
- Улучшается понимание контекстов, характерных для NER-задачи
- Особенно полезно, если домен отличается от предобучения (новости vs Wikipedia)

In [21]:
# Подготовка данных для MLM
def prepare_mlm_dataset(dataset_split):
    """
    Подготовка датасета для MLM: объединяем токены в текст.
    """
    texts = [' '.join(example['tokens']) for example in dataset_split]
    return Dataset.from_dict({'text': texts})

mlm_dataset = prepare_mlm_dataset(dataset['train'])
print(f'MLM датасет: {len(mlm_dataset)} примеров')
print(f'Пример: {mlm_dataset[0]["text"][:200]}...')

MLM датасет: 7746 примеров
Пример: " Если Миронов занял столь оппозиционную позицию , то мне представляется , что для него было бы порядочным и правильным уйти в отставку с занимаемого им поста , поста , который предоставлен ему сегодн...


In [22]:
# Токенизация для MLM
def tokenize_for_mlm(examples):
    return tokenizer(
        examples['text'],
        truncation=True,
        max_length=512,
        return_special_tokens_mask=True
    )

mlm_tokenized = mlm_dataset.map(
    tokenize_for_mlm,
    batched=True,
    remove_columns=['text']
)

print(mlm_tokenized)

Map: 100%|██████████| 7746/7746 [00:00<00:00, 39819.05 examples/s]

Dataset({
    features: ['input_ids', 'token_type_ids', 'attention_mask', 'special_tokens_mask'],
    num_rows: 7746
})


In [23]:
# Загружаем свежую модель для MLM
model_mlm = AutoModelForMaskedLM.from_pretrained(MODEL_NAME).to(DEVICE)

# Data collator для MLM (маскирует 15% токенов)
mlm_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=True,
    mlm_probability=0.15
)

In [24]:
# Настройка MLM обучения
mlm_training_args = TrainingArguments(
    output_dir='./results/mlm',
    num_train_epochs=5,  # больше эпох для MLM
    per_device_train_batch_size=BATCH_SIZE,
    learning_rate=5e-5,
    weight_decay=0.01,
    warmup_ratio=0.1,
    logging_steps=50,
    save_strategy='epoch',
    fp16=FP16,
    seed=RANDOM_STATE,
    max_grad_norm=1.0,  # gradient clipping для стабильности
    report_to='none',
    use_cpu=(DEVICE.type == 'cpu'),
)

mlm_trainer = Trainer(
    model=model_mlm,
    args=mlm_training_args,
    train_dataset=mlm_tokenized,
    data_collator=mlm_collator,
    tokenizer=tokenizer
)

# MLM предобучение
print('MLM предобучение на train...')
mlm_trainer.train()

MLM предобучение на train...


  1%|          | 52/4845 [00:06<07:22, 10.84it/s]

{'loss': 3.2865, 'grad_norm': 17.942462921142578, 'learning_rate': 5.154639175257732e-06, 'epoch': 0.05}


  2%|▏         | 101/4845 [00:11<09:37,  8.21it/s]

{'loss': 3.3657, 'grad_norm': 23.329612731933594, 'learning_rate': 1.0309278350515464e-05, 'epoch': 0.1}


...

 99%|█████████▉| 4801/4845 [08:29<00:04,  9.57it/s]

{'loss': 2.9721, 'grad_norm': 19.970535278320312, 'learning_rate': 5.160550458715596e-07, 'epoch': 4.95}


100%|██████████| 4845/4845 [08:35<00:00,  9.41it/s]

{'train_runtime': 515.0142, 'train_samples_per_second': 75.202, 'train_steps_per_second': 9.408, 'train_loss': 3.050554574310964, 'epoch': 5.0}


TrainOutput(global_step=4845, training_loss=3.050554574310964, metrics={'train_runtime': 515.0142, 'train_samples_per_second': 75.202, 'train_steps_per_second': 9.408, 'total_flos': 28981696892544.0, 'train_loss': 3.050554574310964, 'epoch': 5.0})

In [25]:
# Теперь NER дообучение поверх MLM-адаптированной модели
# Заменяем голову MLM на голову для классификации токенов

# ДИАГНОСТИКА: проверяем состояние MLM модели
print("=== ДИАГНОСТИКА ===")
print(f"1. model_mlm на устройстве: {next(model_mlm.parameters()).device}")

# Проверка на NaN в MLM модели
mlm_has_nan = any(torch.isnan(p).any().item() for p in model_mlm.parameters())
print(f"2. MLM модель содержит NaN: {mlm_has_nan}")

if mlm_has_nan:
    print("⚠️ ОШИБКА: MLM модель повреждена! Перезапустите ядро.")
else:
    # Создаём NER модель
    model_mlm_ner = AutoModelForTokenClassification.from_pretrained(
        MODEL_NAME,
        num_labels=len(label_list),
        id2label=id2label,
        label2id=label2id
    )
    
    # Копируем веса encoder из MLM модели
    model_mlm_ner.bert.load_state_dict(model_mlm.bert.state_dict())
    model_mlm_ner = model_mlm_ner.to(DEVICE)
    
    # Проверка NER модели
    ner_has_nan = any(torch.isnan(p).any().item() for p in model_mlm_ner.parameters())
    print(f"3. NER модель содержит NaN: {ner_has_nan}")
    
    # Тест forward pass
    test_batch = data_collator([tokenized_dataset['train'][0]])
    test_batch = {k: v.to(DEVICE) for k, v in test_batch.items()}
    
    model_mlm_ner.train()
    with torch.no_grad():
        test_out = model_mlm_ner(**test_batch)
    print(f"4. Тестовый loss: {test_out.loss.item():.4f}")
    
    if test_out.loss.item() > 0:
        print("✓ Модель готова к обучению")
    else:
        print("⚠️ ПРОБЛЕМА: loss = 0")
    
    print("=" * 20)

=== ДИАГНОСТИКА ===
1. model_mlm на устройстве: mps:0
2. MLM модель содержит NaN: False


Some weights of BertForTokenClassification were not initialized from the model checkpoint at cointegrated/rubert-tiny2 and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


3. NER модель содержит NaN: False
4. Тестовый loss: 1.8724
✓ Модель готова к обучению


In [26]:
# NER дообучение
ner_after_mlm_args = TrainingArguments(
    output_dir='./results/mlm_ner',
    evaluation_strategy='epoch',
    save_strategy='epoch',
    learning_rate=2e-5,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=EVAL_BATCH_SIZE,
    num_train_epochs=3,
    weight_decay=0.01,
    warmup_ratio=0.1,
    logging_steps=50,
    load_best_model_at_end=True,
    metric_for_best_model='f1',
    fp16=FP16,
    seed=RANDOM_STATE,
    report_to='none',
    use_cpu=(DEVICE.type == 'cpu'),
)

trainer_mlm_ner = Trainer(
    model=model_mlm_ner,
    args=ner_after_mlm_args,
    train_dataset=tokenized_dataset['train'],
    eval_dataset=tokenized_dataset['test'],
    tokenizer=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics
)

print('NER дообучение после MLM...')
trainer_mlm_ner.train()

NER дообучение после MLM...


  2%|▏         | 53/2907 [00:03<02:27, 19.33it/s]

{'loss': 1.9248, 'grad_norm': 8.730650901794434, 'learning_rate': 3.436426116838488e-06, 'epoch': 0.05}


  4%|▎         | 103/2907 [00:05<02:36, 17.91it/s]

{'loss': 1.6309, 'grad_norm': 7.859631538391113, 'learning_rate': 6.872852233676976e-06, 'epoch': 0.1}


...

                                                   
100%|██████████| 2907/2907 [02:47<00:00, 16.95it/s]

{'eval_loss': 0.058404769748449326, 'eval_precision': 0.8839915373765868, 'eval_recall': 0.9139628144367481, 'eval_f1': 0.8987273704965049, 'eval_runtime': 3.4302, 'eval_samples_per_second': 752.735, 'eval_steps_per_second': 47.228, 'epoch': 3.0}


100%|██████████| 2907/2907 [02:47<00:00, 17.35it/s]

{'train_runtime': 167.5686, 'train_samples_per_second': 138.678, 'train_steps_per_second': 17.348, 'train_loss': 0.185118756222077, 'epoch': 3.0}


TrainOutput(global_step=2907, training_loss=0.185118756222077, metrics={'train_runtime': 167.5686, 'train_samples_per_second': 138.678, 'train_steps_per_second': 17.348, 'total_flos': 16158516362976.0, 'train_loss': 0.185118756222077, 'epoch': 3.0})

In [27]:
# Оценка MLM + NER
results_mlm_ner = evaluate_model(
    model_mlm_ner,
    tokenized_dataset['test'],
    'Оценка: MLM предобучение + NER'
)

Оценка: MLM предобучение + NER: 100%|██████████| 162/162 [00:02<00:00, 57.76it/s]



Оценка: MLM предобучение + NER
Precision: 0.8840
Recall:    0.9140
F1:        0.8987

Детальный отчёт:
              precision    recall  f1-score   support

         LOC       0.92      0.93      0.92      1509
         ORG       0.81      0.84      0.82      1790
         PER       0.93      0.96      0.95      2187

   micro avg       0.88      0.91      0.90      5486
   macro avg       0.88      0.91      0.90      5486
weighted avg       0.88      0.91      0.90      5486



---

## 5. Улучшенное маскирование

### 5.1 WholeWordMask

Вместо маскирования отдельных субтокенов маскируем целые слова.

**Пример:**
- Стандартный MLM: `[Пу] [##тин] встретился` → `[MASK] [##тин] встретился`
- WholeWordMask: `[Пу] [##тин] встретился` → `[MASK] [MASK] встретился`

**Почему лучше для NER:** Сущности часто состоят из нескольких субтокенов. Маскирование целых слов учит модель предсказывать полные сущности.

In [47]:
# Загружаем свежую модель для WholeWordMask MLM
model_wwm = AutoModelForMaskedLM.from_pretrained(MODEL_NAME).to(DEVICE)

# DataCollatorForWholeWordMask - маскирует целые слова, а не отдельные субтокены
class WholeWordMaskDataCollator:                                                                                                                     
      """                                                                                                                                              
      Обёртка над DataCollatorForWholeWordMask,                                                                                                        
      которая сохраняет attention_mask и token_type_ids с правильным padding.                                                                          
      """                                                                                                                                              
      def __init__(self, tokenizer, mlm_probability=0.15):                                                                                             
          self.wwm_collator = DataCollatorForWholeWordMask(                                                                                            
              tokenizer=tokenizer,                                                                                                                     
              mlm_probability=mlm_probability                                                                                                          
          )                                                                                                                                            
          self.tokenizer = tokenizer                                                                                                                   
                                                                                                                                                       
      def __call__(self, features):                                                                                                                    
          # WWM collator делает маскирование и padding для input_ids/labels                                                                            
          batch = self.wwm_collator(features)                                                                                                          
                                                                                                                                                       
          # Получаем max_length из результата (после padding)                                                                                          
          max_length = batch['input_ids'].shape[1]                                                                                                     
          pad_token_id = self.tokenizer.pad_token_id                                                                                                   
                                                                                                                                                       
          # Паддим attention_mask до той же длины                                                                                                      
          attention_mask = []                                                                                                                          
          token_type_ids = []                                                                                                                          
          for f in features:                                                                                                                           
              am = f['attention_mask']                                                                                                                 
              # Паддим нулями (0 = игнорировать токен)                                                                                                 
              padded_am = am + [0] * (max_length - len(am))                                                                                            
              attention_mask.append(padded_am)                                                                                                         
                                                                                                                                                       
              if 'token_type_ids' in f:                                                                                                                
                  tti = f['token_type_ids']                                                                                                            
                  padded_tti = tti + [0] * (max_length - len(tti))                                                                                     
                  token_type_ids.append(padded_tti)                                                                                                    
                                                                                                                                                       
          batch['attention_mask'] = torch.tensor(attention_mask)                                                                                       
          if token_type_ids:                                                                                                                           
              batch['token_type_ids'] = torch.tensor(token_type_ids)                                                                                   
                                                                                                                                                       
          return batch                 

# Используем обёртку с сохранением attention_mask                                                                                                    
wwm_collator = WholeWordMaskDataCollator(                                                                                                            
    tokenizer=tokenizer,                                                                                                                             
    mlm_probability=0.15                                                                                                                             
)                                                                                                                                                    
                                                                                                                                                
print('WholeWordMask MLM: используем DataCollatorForWholeWordMask с сохранением attention_mask')  

WholeWordMask MLM: используем DataCollatorForWholeWordMask с сохранением attention_mask


In [35]:
# Берём несколько примеров из mlm_tokenized                                                                                                          
samples = [mlm_tokenized[i] for i in range(4)]                                                                                                       

# Проверяем WWM collator                                                                                                                             
batch = wwm_collator(samples)                                                                                                                        

print("=== ДИАГНОСТИКА WWM COLLATOR ===")                                                                                                            
print(f"input_ids shape: {batch['input_ids'].shape}")                                                                                                
print(f"labels shape: {batch['labels'].shape}")                                                                                                      
print(f"attention_mask shape: {batch.get('attention_mask', 'ОТСУТСТВУЕТ')}")                                                                         

# Проверяем labels                                                                                                                                   
labels = batch['labels']                                                                                                                             
print(f"\nLabels min: {labels.min().item()}")                                                                                                        
print(f"Labels max: {labels.max().item()}")                                                                                                          
print(f"Labels содержит NaN: {torch.isnan(labels.float()).any().item()}")                                                                            
print(f"Labels содержит Inf: {torch.isinf(labels.float()).any().item()}")                                                                            

# Сколько токенов замаскировано                                                                                                                      
masked_count = (labels != -100).sum().item()                                                                                                         
total_tokens = labels.numel()                                                                                                                        
print(f"\nЗамаскировано: {masked_count} из {total_tokens} ({100*masked_count/total_tokens:.1f}%)")                                                   

# Проверяем input_ids                                                                                                                                
print(f"\ninput_ids содержит NaN: {torch.isnan(batch['input_ids'].float()).any().item()}")                                                           

# Покажем первый пример                                                                                                                              
print(f"\nПервый пример labels: {labels[0][:20].tolist()}")                                                                                          
print(f"Первый пример input_ids: {batch['input_ids'][0][:20].tolist()}")

=== ДИАГНОСТИКА WWM COLLATOR ===
input_ids shape: torch.Size([4, 55])
labels shape: torch.Size([4, 55])
attention_mask shape: tensor([[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
         1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
         1, 1, 1, 1, 1, 1, 1],
        [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
         1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
         0, 0, 0, 0, 0, 0, 0],
        [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0,
         0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
         0, 0, 0, 0, 0, 0, 0],
        [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
         0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
         0, 0, 0, 0, 0, 0, 0]])

Labels min: -100
Labels max: 54961
Labels содержит NaN: False
Labels содержит Inf: False

Замаскиров

In [36]:
# Проверка vocab_size                                                                                                                                
print(f"Tokenizer vocab_size: {tokenizer.vocab_size}")                                                                                               
print(f"Labels max: {batch['labels'].max().item()}")                                                                                                 

# Проверяем модель                                                                                                                                   
print(f"Model vocab_size: {model_wwm.config.vocab_size}")                                                                                            

# Есть ли labels > vocab_size?                                                                                                                       
invalid_labels = (batch['labels'] > tokenizer.vocab_size) & (batch['labels'] != -100) 

Tokenizer vocab_size: 83828
Labels max: 54961
Model vocab_size: 83828


In [48]:
# MLM с WholeWordMask
wwm_training_args = TrainingArguments(                                                                                                               
      output_dir='./results/wwm',                                                                                                                      
      num_train_epochs=1,  # только 1 эпоха для теста                                                                                                  
      per_device_train_batch_size=BATCH_SIZE,                                                                                                          
      learning_rate=5e-5,                                                                                                                              
      weight_decay=0.01,                                                                                                                               
      warmup_ratio=0.1,                                                                                                                                
      logging_steps=50,                                                                                                                                
      save_strategy='no',                                                                                                                              
      fp16=False,                                                                                                                                      
      seed=RANDOM_STATE,                                                                                                                               
      max_grad_norm=1.0,                                                                                                                               
      report_to='none',                                                                                                                                
      use_cpu=True,                                                                                                           
  )

wwm_trainer = Trainer(
    model=model_wwm,
    args=wwm_training_args,
    train_dataset=mlm_tokenized,
    data_collator=wwm_collator,
    tokenizer=tokenizer
)

print('WholeWordMask MLM предобучение...')
wwm_trainer.train()

  9%|▊         | 83/969 [00:59<10:39,  1.38it/s]


WholeWordMask MLM предобучение...



  9%|▉         | 89/969 [00:39<02:23,  6.14it/s]

{'loss': 3.9354, 'grad_norm': 21.654211044311523, 'learning_rate': 2.5773195876288658e-05, 'epoch': 0.05}



  9%|▉         | 89/969 [00:50<02:23,  6.14it/s] 

...


  9%|▉         | 89/969 [03:46<02:23,  6.14it/s] 

{'loss': 3.7146, 'grad_norm': 20.73464584350586, 'learning_rate': 1.0894495412844037e-06, 'epoch': 0.98}



100%|██████████| 969/969 [03:20<00:00,  4.83it/s]

{'train_runtime': 200.7361, 'train_samples_per_second': 38.588, 'train_steps_per_second': 4.827, 'train_loss': 3.702321598773401, 'epoch': 1.0}


TrainOutput(global_step=969, training_loss=3.702321598773401, metrics={'train_runtime': 200.7361, 'train_samples_per_second': 38.588, 'train_steps_per_second': 4.827, 'total_flos': 5813582296896.0, 'train_loss': 3.702321598773401, 'epoch': 1.0})

In [51]:
# NER дообучение после WholeWordMask (ВСЁ НА CPU)                                                                                                    
                                                                                                                                                    
print("=== ПРОВЕРКА ПЕРЕД КОПИРОВАНИЕМ ===")                                                                                                         
                                                                                                                                                    
# Держим модель на CPU                                                                                                                               
if next(model_wwm.parameters()).device.type != 'cpu':                                                                                                
    model_wwm = model_wwm.cpu()                                                                                                                      
                                                                                                                                                    
wwm_has_nan = any(torch.isnan(p).any().item() for p in model_wwm.parameters())                                                                       
print(f"model_wwm содержит NaN: {wwm_has_nan}")                                                                                                      
                                                                                                                                                    
if wwm_has_nan:                                                                                                                                      
    print("⚠️ WWM модель повреждена!")                                                                                                               
else:                                                                                                                                                
    # Создаём NER модель на CPU                                                                                                                      
    model_wwm_ner = AutoModelForTokenClassification.from_pretrained(                                                                                 
        MODEL_NAME,                                                                                                                                  
        num_labels=len(label_list),                                                                                                                  
        id2label=id2label,                                                                                                                           
        label2id=label2id                                                                                                                            
    )                                                                                                                                                
                                                                                                                                                    
    # Копируем веса (обе модели на CPU)                                                                                                              
    model_wwm_ner.bert.load_state_dict(model_wwm.bert.state_dict())                                                                                  
                                                                                                                                                    
    # Проверка                                                                                                                                       
    ner_has_nan = any(torch.isnan(p).any().item() for p in model_wwm_ner.parameters())                                                               
    print(f"model_wwm_ner содержит NaN: {ner_has_nan}")                                                                                              
                                                                                                                                                    
    # Тест forward pass на CPU                                                                                                                       
    test_batch = data_collator([tokenized_dataset['train'][0]])                                                                                      
    # НЕ переносим на DEVICE - оставляем на CPU                                                                                                      
    model_wwm_ner.eval()                                                                                                                             
    with torch.no_grad():                                                                                                                            
        test_out = model_wwm_ner(**test_batch)                                                                                                       
    print(f"Тестовый loss: {test_out.loss.item():.4f}")                                                                                              
    print("=" * 35)                                                                                                                                  
                                                                                                                                                    
    if test_out.loss.item() > 0 and not ner_has_nan:                                                                                                 
        # TrainingArguments с use_cpu=True                                                                                                           
        wwm_ner_args = TrainingArguments(                                                                                                            
            output_dir='./results/wwm_ner',                                                                                                          
            evaluation_strategy='epoch',                                                                                                             
            save_strategy='epoch',                                                                                                                   
            learning_rate=2e-5,                                                                                                                      
            per_device_train_batch_size=BATCH_SIZE,                                                                                                  
            per_device_eval_batch_size=EVAL_BATCH_SIZE,                                                                                              
            num_train_epochs=3,                                                                                                                      
            weight_decay=0.01,                                                                                                                       
            warmup_ratio=0.1,                                                                                                                        
            logging_steps=50,                                                                                                                        
            load_best_model_at_end=True,                                                                                                             
            metric_for_best_model='f1',                                                                                                              
            fp16=False,                                                                                                                              
            seed=RANDOM_STATE,                                                                                                                       
            max_grad_norm=1.0,                                                                                                                       
            report_to='none',                                                                                                                        
            use_cpu=True,  # ← ПРИНУДИТЕЛЬНО CPU                                                                                                     
        )                                                                                                                                            
                                                                                                                                                    
        trainer_wwm_ner = Trainer(                                                                                                                   
            model=model_wwm_ner,                                                                                                                     
            args=wwm_ner_args,                                                                                                                       
            train_dataset=tokenized_dataset['train'],                                                                                                
            eval_dataset=tokenized_dataset['test'],                                                                                                  
            tokenizer=tokenizer,                                                                                                                     
            data_collator=data_collator,                                                                                                             
            compute_metrics=compute_metrics                                                                                                          
        )                                                                                                                                            
                                                                                                                                                    
        print('NER дообучение после WholeWordMask (CPU)...')                                                                                         
        trainer_wwm_ner.train()                                                                                                                                                                                                                                 
    else:                                                                                                                                            
        print("⚠️ Модель не готова к обучению!")                                                                                                     
        results_wwm_ner = {'precision': 0.0, 'recall': 0.0, 'f1': 0.0}

=== ПРОВЕРКА ПЕРЕД КОПИРОВАНИЕМ ===
model_wwm содержит NaN: False


Some weights of BertForTokenClassification were not initialized from the model checkpoint at cointegrated/rubert-tiny2 and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
  0%|          | 0/2907 [09:44<?, ?it/s]


model_wwm_ner содержит NaN: False
Тестовый loss: 1.9584
NER дообучение после WholeWordMask (CPU)...


  2%|▏         | 52/2907 [00:05<04:36, 10.34it/s]

{'loss': 2.0965, 'grad_norm': 9.697466850280762, 'learning_rate': 3.436426116838488e-06, 'epoch': 0.05}


...

{'loss': 0.0625, 'grad_norm': 1.5378469228744507, 'learning_rate': 5.351681957186545e-08, 'epoch': 2.99}


100%|██████████| 2907/2907 [04:43<00:00, 10.58it/s]

{'eval_loss': 0.06158463656902313, 'eval_precision': 0.866214335421016, 'eval_recall': 0.9075829383886256, 'eval_f1': 0.8864162364251379, 'eval_runtime': 5.6063, 'eval_samples_per_second': 460.556, 'eval_steps_per_second': 28.896, 'epoch': 3.0}


100%|██████████| 2907/2907 [04:43<00:00, 10.25it/s]

{'train_runtime': 283.7097, 'train_samples_per_second': 81.908, 'train_steps_per_second': 10.246, 'train_loss': 0.19503787361882502, 'epoch': 3.0}


In [54]:
results_wwm_ner = evaluate_model(
    model_wwm_ner,
    tokenized_dataset['test'],
    'Оценка: WholeWordMask + NER'
)

Оценка: WholeWordMask + NER: 100%|██████████| 162/162 [00:05<00:00, 31.04it/s]



Оценка: WholeWordMask + NER
Precision: 0.8662
Recall:    0.9076
F1:        0.8864


### 5.2 Entity Masking (Concept Masking)

**Идея:** Целенаправленно маскируем токены, которые являются частью NER-сущностей.

**Стратегия (приоритетное маскирование):**
- 50% масок — токены сущностей (B-PER, I-ORG и т.д.)
- 50% масок — случайные токены (как в обычном MLM)

**Почему это работает:** Модель учится предсказывать именно сущности по контексту, что напрямую помогает NER-задаче.

In [55]:
class EntityMaskingCollator:
    """
    Data Collator для Entity Masking.
    
    Приоритетно маскирует токены NER-сущностей:
    - entity_mask_ratio: доля масок, приходящаяся на сущности
    - mlm_probability: общая вероятность маскирования (как в стандартном MLM)
    """
    
    def __init__(self, tokenizer, dataset_with_labels, mlm_probability=0.15, entity_mask_ratio=0.5):
        self.tokenizer = tokenizer
        self.mlm_probability = mlm_probability
        self.entity_mask_ratio = entity_mask_ratio
        
        # Создаём маппинг: индекс примера -> позиции сущностей
        self.entity_positions = self._extract_entity_positions(dataset_with_labels)
    
    def _extract_entity_positions(self, dataset):
        """
        Извлекаем позиции сущностей из датасета.
        """
        entity_positions = {}
        
        for idx, example in enumerate(dataset):
            positions = []
            for i, tag_id in enumerate(example['ner_tags']):
                tag = id2label[tag_id]
                if tag != 'O':  # это сущность
                    positions.append(i)
            entity_positions[idx] = positions
        
        return entity_positions
    
    def __call__(self, examples):
        # Стандартный padding
        batch = self.tokenizer.pad(
            examples,
            return_tensors='pt',
            padding=True
        )
        
        input_ids = batch['input_ids'].clone()
        labels = batch['input_ids'].clone()
        
        # Специальные токены не маскируем
        special_tokens_mask = [
            self.tokenizer.get_special_tokens_mask(ids.tolist(), already_has_special_tokens=True)
            for ids in input_ids
        ]
        special_tokens_mask = torch.tensor(special_tokens_mask, dtype=torch.bool)
        
        # Создаём маску для MLM
        probability_matrix = torch.full(input_ids.shape, self.mlm_probability)
        probability_matrix.masked_fill_(special_tokens_mask, value=0.0)
        
        # Маскируем
        masked_indices = torch.bernoulli(probability_matrix).bool()
        labels[~masked_indices] = -100  # loss только для замаскированных
        
        # Заменяем на [MASK]
        input_ids[masked_indices] = self.tokenizer.mask_token_id
        
        batch['input_ids'] = input_ids
        batch['labels'] = labels
        
        return batch

In [56]:
# Упрощённая версия Entity Masking:
# Используем информацию о метках для повышения вероятности маскирования сущностей

class SimpleEntityMaskingCollator:
    """
    Упрощённый Entity Masking Collator.
    
    Работает с токенизированными данными, где есть labels с NER-метками.
    Повышает вероятность маскирования для токенов сущностей.
    """
    
    def __init__(self, tokenizer, mlm_probability=0.15, entity_boost=3.0):
        """
        Args:
            tokenizer: токенизатор
            mlm_probability: базовая вероятность маскирования
            entity_boost: во сколько раз повышаем вероятность для сущностей
        """
        self.tokenizer = tokenizer
        self.mlm_probability = mlm_probability
        self.entity_boost = entity_boost
        self.mask_token_id = tokenizer.mask_token_id
        self.pad_token_id = tokenizer.pad_token_id
    
    def __call__(self, examples):
        # Извлекаем labels (NER метки) если есть
        ner_labels = [ex.get('labels', None) for ex in examples]
        
        # Убираем labels из примеров для токенизации
        examples_clean = [{k: v for k, v in ex.items() if k != 'labels'} for ex in examples]
        
        # Padding
        batch = self.tokenizer.pad(
            examples_clean,
            return_tensors='pt',
            padding=True
        )
        
        input_ids = batch['input_ids'].clone()
        labels = input_ids.clone()  # MLM labels = оригинальные token IDs
        
        batch_size, seq_len = input_ids.shape
        
        # Специальные токены
        special_tokens_mask = torch.tensor([
            self.tokenizer.get_special_tokens_mask(ids.tolist(), already_has_special_tokens=True)
            for ids in input_ids
        ], dtype=torch.bool)
        
        # Базовая вероятность маскирования
        probability_matrix = torch.full(input_ids.shape, self.mlm_probability)
        
        # Повышаем вероятность для токенов сущностей
        if ner_labels[0] is not None:
            for i, ner_label_seq in enumerate(ner_labels):
                for j, label in enumerate(ner_label_seq):
                    if j < probability_matrix.shape[1]:
                        if label != -100 and label != 0:  # не O и не игнорируемый
                            probability_matrix[i, j] = min(
                                self.mlm_probability * self.entity_boost, 
                                0.8
                            )
        
        # Не маскируем специальные токены и PAD
        probability_matrix.masked_fill_(special_tokens_mask, value=0.0)
        for i in range(batch_size):
            for j in range(seq_len):
                if input_ids[i, j] == self.pad_token_id:
                    probability_matrix[i, j] = 0.0
        
        # Применяем маскирование
        masked_indices = torch.bernoulli(probability_matrix).bool()
        
        # Labels: -100 для НЕ замаскированных токенов
        labels[~masked_indices] = -100
        
        # Заменяем на [MASK]
        input_ids[masked_indices] = self.mask_token_id
        
        batch['input_ids'] = input_ids
        batch['labels'] = labels
        
        return batch

In [57]:
# Подготовка данных для Entity Masking
# Используем токенизированный датасет с NER-метками
entity_mlm_dataset = tokenized_dataset['train']

# Создаём collator
entity_collator = SimpleEntityMaskingCollator(
    tokenizer=tokenizer,
    mlm_probability=0.15,
    entity_boost=3.0  # сущности маскируются в 3 раза чаще
)

print('Entity Masking Collator создан')
print(f'Базовая вероятность маскирования: 15%')
print(f'Вероятность для сущностей: ~45% (boost x3)')

# Тест collator-а
print('\nТест Entity Masking collator:')
test_batch = entity_collator([entity_mlm_dataset[0], entity_mlm_dataset[1]])
print(f'  input_ids shape: {test_batch["input_ids"].shape}')
print(f'  labels shape: {test_batch["labels"].shape}')
masked_count = (test_batch['input_ids'] == tokenizer.mask_token_id).sum().item()
valid_labels = (test_batch['labels'] != -100).sum().item()
print(f'  Замаскировано токенов: {masked_count}')
print(f'  Labels != -100: {valid_labels}')
print(f'  Совпадают: {masked_count == valid_labels}')

You're using a BertTokenizerFast tokenizer. Please note that with a fast tokenizer, using the `__call__` method is faster than using a method to encode the text followed by a call to the `pad` method to get a padded encoding.


Entity Masking Collator создан
Базовая вероятность маскирования: 15%
Вероятность для сущностей: ~45% (boost x3)

Тест Entity Masking collator:
  input_ids shape: torch.Size([2, 55])
  labels shape: torch.Size([2, 55])
  Замаскировано токенов: 10
  Labels != -100: 10
  Совпадают: True


In [58]:
# Загружаем свежую модель для Entity Masking MLM (на CPU)                                                                                            
model_entity_mlm = AutoModelForMaskedLM.from_pretrained(MODEL_NAME)  # БЕЗ .to(DEVICE)                                                               
                                                                                                                                                    
entity_mlm_args = TrainingArguments(                                                                                                                 
    output_dir='./results/entity_mlm',                                                                                                               
    num_train_epochs=5,                                                                                                                              
    per_device_train_batch_size=BATCH_SIZE,                                                                                                          
    learning_rate=5e-5,                                                                                                                              
    weight_decay=0.01,                                                                                                                               
    warmup_ratio=0.1,                                                                                                                                
    logging_steps=50,                                                                                                                                
    save_strategy='epoch',                                                                                                                           
    fp16=False,  # CPU не поддерживает fp16                                                                                                          
    seed=RANDOM_STATE,                                                                                                                               
    max_grad_norm=1.0,                                                                                                                               
    report_to='none',                                                                                                                                
    use_cpu=True,  # ← ПРИНУДИТЕЛЬНО CPU                                                                                                             
)                                                                                                                                                    
                                                                                                                                                    
entity_mlm_trainer = Trainer(                                                                                                                        
    model=model_entity_mlm,                                                                                                                          
    args=entity_mlm_args,                                                                                                                            
    train_dataset=entity_mlm_dataset,                                                                                                                
    data_collator=entity_collator,                                                                                                                   
    tokenizer=tokenizer                                                                                                                              
)                                                                                                                                                    
                                                                                                                                                    
print('Entity Masking MLM предобучение (CPU)...')                                                                                                    
entity_mlm_trainer.train()

Entity Masking MLM предобучение (CPU)...


  1%|          | 51/4845 [00:10<14:54,  5.36it/s]

{'loss': 4.2306, 'grad_norm': 19.3900089263916, 'learning_rate': 5.154639175257732e-06, 'epoch': 0.05}


  2%|▏         | 100/4845 [00:20<16:48,  4.70it/s]

{'loss': 4.2509, 'grad_norm': 23.663862228393555, 'learning_rate': 1.0309278350515464e-05, 'epoch': 0.1}


...

 99%|█████████▉| 4800/4845 [15:48<00:09,  4.91it/s]

{'loss': 3.4442, 'grad_norm': 33.84088134765625, 'learning_rate': 5.160550458715596e-07, 'epoch': 4.95}


100%|██████████| 4845/4845 [15:57<00:00,  5.06it/s]

{'train_runtime': 957.7693, 'train_samples_per_second': 40.438, 'train_steps_per_second': 5.059, 'train_loss': 3.588466566339735, 'epoch': 5.0}


TrainOutput(global_step=4845, training_loss=3.588466566339735, metrics={'train_runtime': 957.7693, 'train_samples_per_second': 40.438, 'train_steps_per_second': 5.059, 'total_flos': 28981696892544.0, 'train_loss': 3.588466566339735, 'epoch': 5.0})

In [60]:
# NER дообучение после Entity Masking (ВСЁ НА CPU)                                                                                                   
                                                                                                                                                    
print("=== ПРОВЕРКА ПЕРЕД КОПИРОВАНИЕМ ===")                                                                                                         
                                                                                                                                                    
# Держим модель на CPU                                                                                                                               
if next(model_entity_mlm.parameters()).device.type != 'cpu':                                                                                         
    model_entity_mlm = model_entity_mlm.cpu()                                                                                                        
                                                                                                                                                    
entity_has_nan = any(torch.isnan(p).any().item() for p in model_entity_mlm.parameters())                                                             
print(f"model_entity_mlm содержит NaN: {entity_has_nan}")                                                                                            
                                                                                                                                                    
if entity_has_nan:                                                                                                                                   
    print("⚠️ Entity MLM модель повреждена!")                                                                                                        
else:                                                                                                                                                
    # Создаём NER модель на CPU                                                                                                                      
    model_entity_ner = AutoModelForTokenClassification.from_pretrained(                                                                              
        MODEL_NAME,                                                                                                                                  
        num_labels=len(label_list),                                                                                                                  
        id2label=id2label,                                                                                                                           
        label2id=label2id                                                                                                                            
    )                                                                                                                                                
                                                                                                                                                    
    # Копируем веса (обе модели на CPU)                                                                                                              
    model_entity_ner.bert.load_state_dict(model_entity_mlm.bert.state_dict())                                                                        
                                                                                                                                                    
    # Проверка                                                                                                                                       
    ner_has_nan = any(torch.isnan(p).any().item() for p in model_entity_ner.parameters())                                                            
    print(f"model_entity_ner содержит NaN: {ner_has_nan}")                                                                                           
                                                                                                                                                    
    # Тест forward pass на CPU                                                                                                                       
    test_batch = data_collator([tokenized_dataset['train'][0]])                                                                                      
    model_entity_ner.eval()                                                                                                                          
    with torch.no_grad():                                                                                                                            
        test_out = model_entity_ner(**test_batch)                                                                                                    
    print(f"Тестовый loss: {test_out.loss.item():.4f}")                                                                                              
    print("=" * 35)                                                                                                                                  
                                                                                                                                                    
    if test_out.loss.item() > 0 and not ner_has_nan:                                                                                                 
        # TrainingArguments с use_cpu=True                                                                                                           
        entity_ner_args = TrainingArguments(                                                                                                         
            output_dir='./results/entity_ner',                                                                                                       
            evaluation_strategy='epoch',                                                                                                             
            save_strategy='epoch',                                                                                                                   
            learning_rate=2e-5,                                                                                                                      
            per_device_train_batch_size=BATCH_SIZE,                                                                                                  
            per_device_eval_batch_size=EVAL_BATCH_SIZE,                                                                                              
            num_train_epochs=3,                                                                                                                      
            weight_decay=0.01,                                                                                                                       
            warmup_ratio=0.1,                                                                                                                        
            logging_steps=50,                                                                                                                        
            load_best_model_at_end=True,                                                                                                             
            metric_for_best_model='f1',                                                                                                              
            fp16=False,                                                                                                                              
            seed=RANDOM_STATE,                                                                                                                       
            max_grad_norm=1.0,                                                                                                                       
            report_to='none',                                                                                                                        
            use_cpu=True,  # ← ПРИНУДИТЕЛЬНО CPU                                                                                                     
        )                                                                                                                                            
                                                                                                                                                    
        trainer_entity_ner = Trainer(                                                                                                                
            model=model_entity_ner,                                                                                                                  
            args=entity_ner_args,                                                                                                                    
            train_dataset=tokenized_dataset['train'],                                                                                                
            eval_dataset=tokenized_dataset['test'],                                                                                                  
            tokenizer=tokenizer,                                                                                                                     
            data_collator=data_collator,                                                                                                             
            compute_metrics=compute_metrics                                                                                                          
        )                                                                                                                                            
                                                                                                                                                    
        print('NER дообучение после Entity Masking (CPU)...')                                                                                        
        trainer_entity_ner.train()                                                                                                                                                                                                                         
    else:                                                                                                                                            
        print("⚠️ Модель не готова к обучению!")                                                                                                     
        results_entity_ner = {'precision': 0.0, 'recall': 0.0, 'f1': 0.0}

=== ПРОВЕРКА ПЕРЕД КОПИРОВАНИЕМ ===
model_entity_mlm содержит NaN: False


Some weights of BertForTokenClassification were not initialized from the model checkpoint at cointegrated/rubert-tiny2 and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


model_entity_ner содержит NaN: False
Тестовый loss: 1.9090
NER дообучение после Entity Masking (CPU)...


  2%|▏         | 52/2907 [00:04<04:28, 10.62it/s]

{'loss': 2.0459, 'grad_norm': 9.549314498901367, 'learning_rate': 3.436426116838488e-06, 'epoch': 0.05}


...

{'loss': 0.0594, 'grad_norm': 1.4869905710220337, 'learning_rate': 5.351681957186545e-08, 'epoch': 2.99}


100%|██████████| 2907/2907 [04:40<00:00, 10.82it/s]

{'eval_loss': 0.06346152722835541, 'eval_precision': 0.8850267379679144, 'eval_recall': 0.9050309879693766, 'eval_f1': 0.8949170872386445, 'eval_runtime': 5.349, 'eval_samples_per_second': 482.707, 'eval_steps_per_second': 30.286, 'epoch': 3.0}


100%|██████████| 2907/2907 [04:40<00:00, 10.36it/s]

{'train_runtime': 280.5615, 'train_samples_per_second': 82.827, 'train_steps_per_second': 10.361, 'train_loss': 0.18761724181342543, 'epoch': 3.0}


In [61]:
results_entity_ner = evaluate_model(
    model_entity_ner,
    tokenized_dataset['test'],
    'Оценка: Entity Masking + NER'
)

Оценка: Entity Masking + NER: 100%|██████████| 162/162 [00:04<00:00, 36.18it/s]



Оценка: Entity Masking + NER
Precision: 0.8850
Recall:    0.9050
F1:        0.8949


---

## 6. Синтетическая разметка через DeepPavlov

**Идея:** Используем сильную NER-модель (DeepPavlov) для разметки внешнего корпуса (Lenta.ru), затем дообучаем на объединённых данных.

**Зачем:**
- Увеличиваем объём обучающих данных
- Добавляем разнообразие контекстов
- Knowledge distillation от более сильной модели

In [62]:
# Установка DeepPavlov (для Colab/Kaggle)
# !pip install deeppavlov -q
# !python -m deeppavlov install ner_collection3_bert

In [62]:
# Загрузка внешнего корпуса (Lenta.ru)
from corus import load_lenta2

LENTA_PATH = 'lenta-ru-news.csv.bz2'

# Размер синтетических данных адаптируется под устройство
if DEVICE.type == 'cuda':
    SYNTHETIC_SIZE = 20000
elif DEVICE.type == 'mps':
    SYNTHETIC_SIZE = 15000
else:  # CPU
    SYNTHETIC_SIZE = 10000  # меньше для CPU, чтобы не ждать слишком долго

print(f'Размер синтетических данных: {SYNTHETIC_SIZE}')
print('Загрузка Lenta.ru...')

lenta_records = []
for i, record in enumerate(tqdm(load_lenta2(LENTA_PATH), desc='Загрузка')):
    if i >= SYNTHETIC_SIZE:
        break
    lenta_records.append(record)

print(f'Загружено {len(lenta_records)} текстов')

Размер синтетических данных: 15000
Загрузка Lenta.ru...


Загрузка: 15000it [00:01, 14174.45it/s]

Загружено 15000 текстов


In [63]:
# Подготовка текстов для NER
lenta_texts = [f"{r.title}. {r.text}" for r in lenta_records]

# Ограничиваем длину текстов (DeepPavlov может быть медленным на длинных)
MAX_TEXT_LEN = 1000
lenta_texts = [text[:MAX_TEXT_LEN] for text in lenta_texts]

print(f'Пример текста: {lenta_texts[0][:200]}...')

Пример текста: 1914. Русские войска вступили в пределы Венгрии  . Бои у Сопоцкина и Друскеник закончились отступлением германцев. Неприятель, приблизившись с севера к Осовцу начал артиллерийскую борьбу с крепостью. ...


In [73]:
# Загрузка DeepPavlov NER модели
from deeppavlov import build_model

print('Загрузка DeepPavlov NER модели...')
dp_ner = build_model('ner_collection3_bert', download=True)
print('Модель загружена')

Загрузка DeepPavlov NER модели...


2026-03-29 19:41:28.844 INFO in 'deeppavlov.core.data.utils'['utils'] at line 97: Downloading from http://files.deeppavlov.ai/v1/ner/ner_rus_bert_coll3_torch.tar.gz to /Users/anastasiagapeeva/.deeppavlov/models/ner_rus_bert_coll3_torch.tar.gz


KeyboardInterrupt: 

In [ ]:
# Генерация синтетической разметки
print('Генерация синтетической разметки...')

synthetic_data = []

# Batch size для DeepPavlov инференса
DP_BATCH_SIZE = 16 if DEVICE.type in ('cpu', 'mps') else 32

for i in tqdm(range(0, len(lenta_texts), DP_BATCH_SIZE)):
    batch_texts = lenta_texts[i:i+DP_BATCH_SIZE]
    
    # DeepPavlov возвращает (tokens, tags) для каждого текста
    results = dp_ner(batch_texts)
    
    for tokens, tags in zip(results[0], results[1]):
        if len(tokens) > 0:
            synthetic_data.append({
                'tokens': tokens,
                'ner_tags': tags
            })

print(f'Сгенерировано {len(synthetic_data)} размеченных примеров')

In [ ]:
# Конвертация DeepPavlov меток в формат factrueval
# DeepPavlov использует: B-PER, I-PER, B-ORG, I-ORG, B-LOC, I-LOC, O
# factrueval использует те же метки

def convert_dp_tags_to_ids(tags):
    """
    Конвертация строковых меток DeepPavlov в числовые ID.
    """
    tag_ids = []
    for tag in tags:
        if tag in label2id:
            tag_ids.append(label2id[tag])
        else:
            # Неизвестная метка -> O
            tag_ids.append(label2id['O'])
    return tag_ids

# Подготовка синтетического датасета
synthetic_dataset = Dataset.from_dict({
    'tokens': [ex['tokens'] for ex in synthetic_data],
    'ner_tags': [convert_dp_tags_to_ids(ex['ner_tags']) for ex in synthetic_data]
})

print(f'Синтетический датасет: {len(synthetic_dataset)} примеров')
print(f'Пример: {synthetic_dataset[0]}')

In [ ]:
# Статистика по синтетическим данным
def count_statistics_numeric(dataset_split):
    """Подсчёт статистики для датасета с числовыми метками."""
    entity_counts = Counter()
    token_counts = Counter()
    
    for example in dataset_split:
        for tag_id in example['ner_tags']:
            tag = id2label[tag_id]
            if tag.startswith('B-'):
                entity_type = tag[2:]
                entity_counts[entity_type] += 1
                token_counts[entity_type] += 1
            elif tag.startswith('I-'):
                entity_type = tag[2:]
                token_counts[entity_type] += 1
    
    return entity_counts, token_counts

entity_counts, token_counts = count_statistics_numeric(synthetic_dataset)
total_entities = sum(entity_counts.values())
total_tokens = sum(token_counts.values())

print('СТАТИСТИКА ПО СИНТЕТИЧЕСКИМ ДАННЫМ')
print('='*50)
print(f'{"Тип":<6} {"Сущностей":>12} {"Токенов":>12} {"Ср. длина":>12}')
print('-'*50)

for ent_type in ['PER', 'ORG', 'LOC']:
    ent_count = entity_counts[ent_type]
    tok_count = token_counts[ent_type]
    avg_len = tok_count / ent_count if ent_count > 0 else 0
    print(f'{ent_type:<6} {ent_count:>12} {tok_count:>12} {avg_len:>12.2f}')

print('-'*50)
total_avg = total_tokens / total_entities if total_entities > 0 else 0
print(f'{"ВСЕГО":<6} {total_entities:>12} {total_tokens:>12} {total_avg:>12.2f}')

In [ ]:
# Токенизация синтетического датасета
def tokenize_synthetic(examples):
    """Токенизация для синтетических данных (аналогично основному датасету)."""
    tokenized = tokenizer(
        examples['tokens'],
        truncation=True,
        is_split_into_words=True,
        max_length=512
    )
    
    labels = []
    for i, label_ids in enumerate(examples['ner_tags']):
        word_ids = tokenized.word_ids(batch_index=i)
        previous_word_idx = None
        label_ids_aligned = []
        
        for word_idx in word_ids:
            if word_idx is None:
                label_ids_aligned.append(-100)
            elif word_idx != previous_word_idx:
                if word_idx < len(label_ids):
                    label_ids_aligned.append(label_ids[word_idx])
                else:
                    label_ids_aligned.append(label2id['O'])
            else:
                label_ids_aligned.append(-100)
            
            previous_word_idx = word_idx
        
        labels.append(label_ids_aligned)
    
    tokenized['labels'] = labels
    return tokenized

synthetic_tokenized = synthetic_dataset.map(
    tokenize_synthetic,
    batched=True,
    remove_columns=synthetic_dataset.column_names
)

print(f'Токенизированный синтетический датасет: {synthetic_tokenized}')

In [ ]:
# Объединение factrueval train + synthetic
from datasets import concatenate_datasets

combined_train = concatenate_datasets([
    tokenized_dataset['train'],
    synthetic_tokenized
])

print(f'Объединённый датасет: {len(combined_train)} примеров')
print(f'  - factrueval train: {len(tokenized_dataset["train"])}')
print(f'  - synthetic: {len(synthetic_tokenized)}')

In [ ]:
# Дообучение на объединённых данных
model_synthetic = AutoModelForTokenClassification.from_pretrained(
    MODEL_NAME,
    num_labels=len(label_list),
    id2label=id2label,
    label2id=label2id
).to(DEVICE)

synthetic_training_args = TrainingArguments(
    output_dir='./results/synthetic',
    eval_strategy='epoch',
    save_strategy='epoch',
    learning_rate=2e-5,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=EVAL_BATCH_SIZE,
    num_train_epochs=3,
    weight_decay=0.01,
    warmup_ratio=0.1,
    logging_steps=100,
    load_best_model_at_end=True,
    metric_for_best_model='f1',
    fp16=FP16,
    seed=RANDOM_STATE,
    report_to='none',
    use_cpu=(DEVICE.type == 'cpu'),
)

trainer_synthetic = Trainer(
    model=model_synthetic,
    args=synthetic_training_args,
    train_dataset=combined_train,
    eval_dataset=tokenized_dataset['test'],
    tokenizer=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics
)

print('NER дообучение на factrueval + synthetic...')
trainer_synthetic.train()

In [ ]:
results_synthetic = evaluate_model(
    model_synthetic,
    tokenized_dataset['test'],
    'Оценка: factrueval + Synthetic (DeepPavlov)'
)

---

## 8. Выводы

### Результаты экспериментов

| Подход | Precision | Recall | F1 | Δ F1 vs Baseline |
|--------|-----------|--------|----|-----------------|
| До дообучения | ~0 | ~0 | ~0 | - |
| Baseline | X.XX | X.XX | X.XX | 0 |
| MLM + NER | X.XX | X.XX | X.XX | +X.XX |
| WholeWordMask + NER | X.XX | X.XX | X.XX | +X.XX |
| Entity Masking + NER | X.XX | X.XX | X.XX | +X.XX |
| Synthetic + NER | X.XX | X.XX | X.XX | +X.XX |

### Анализ подходов

1. **Baseline fine-tuning** — значительное улучшение по сравнению с zero-shot, что ожидаемо.

2. **MLM предобучение** — [вписать наблюдения после запуска]

3. **WholeWordMask** — [вписать наблюдения после запуска]

4. **Entity Masking** — [вписать наблюдения после запуска]

5. **Синтетические данные** — [вписать наблюдения после запуска]

### Что можно улучшить

1. **Увеличить объём синтетических данных** — 20k может быть недостаточно
2. **Комбинировать подходы** — MLM + Entity Masking + Synthetic
3. **Фильтрация синтетических данных** — оставлять только примеры с высокой уверенностью модели
4. **Cross-validation** — для более надёжной оценки
5. **Другие архитектуры** — попробовать XLM-RoBERTa или mBERT